# Roman Coronagraph Target Detection Probability Pipeline

This tutorial walks through the pipeline for computing detection probabilities for the Roman Coronagraph Instrument's (CGI) top targets.

All targets are RV-detected giant planets with orbital posteriors in one of two formats:
- **RadVel** — RV-only posteriors (periods, eccentricities, semi-amplitudes)
- **Orbitize** — joint RV + Hipparcos-Gaia astrometric posteriors (full 3D orbits)

The pipeline has two stages:

1. **Point cloud generation** — Sample orbital posteriors, propagate orbits over the CGI observing window, and produce a grid of (epoch × posterior draw) with sky-projected separations, phase angles, and planet properties.

2. **Detection probability** — For each posterior draw at each epoch, compute the planet's flux contrast (via the Batalha+2018 atmospheric model grid in `corgidb`, or a Lambert phase function fallback), then use `corgietc` and EXOSIMS to determine the required integration time under the full CGI noise model. Detection probability is reported as the fraction of posterior draws achievable within a given integration time budget.


Let's start by importing all required modules:

In [ ]:
import os, copy
os.environ["CORGIETC_DATA_DIR"] = "/opt/homebrew/anaconda3/envs/roman_orbits/lib/python3.12/site-packages/corgietc/data"

import numpy as np
import pandas as pd
from astropy.time import Time
from astropy import units as u
import matplotlib.pyplot as plt

from roman_orbit_tools import (
    orbit_params, display_name,
    load_posteriors, load_point_cloud,
    gen_point_cloud, gen_summary_csv,
)
from roman_detection_tools import (
    get_planet_config,
    calc_integration_time_for_cloud,
    compute_feasible_pdet,
    compute_observation_windows,
    gen_corgietc_contrast_curves,
    get_iwa_owa
)
from roman_plotting_tools import (
    plot_orbital_parameters,
    plot_contrast_comparison,
)
from corgidb_photometry import (
    load_photometry_grid,
    compute_contrast_picaso,
    compute_contrast_lambert,
)

Let's generate a point cloud (pkl file where all propagated orbital information is stored) for Ups And d from its radvel posteriors:

In [ ]:
# indicate planet and where the posterior is located
planet = "ups_And_d"
# planet = ['ups_And_d', 'eps_Eri_b', '14_Her_b', '47_UMa_c',
#           'HD_154345_b', 'HD_190360_b', 'HD_217107_c', 'HD_114783_c']
posterior_dir = 'orbit_fits/' # for radvel posteriors (RV only)
#posterior_dir = 'orbit_fits/Roman_RV_HGCA_Orbits' # for orbitize posteriors (RV + HGCA)

# make output folder
output_dir = 'outputs/outputs_radvel/'
#output_dir = 'outputs/outputs_orbitize/'
os.makedirs(output_dir, exist_ok=True)

# pick start and end date for point cloud
start_date, end_date = "2026-12-01", "2028-06-30"
# let's note this is the rv only case in the pkl file name
output = f'{planet}_{start_date}_to_{end_date}_RVOnly'

# Load RadVel posterior
df = load_posteriors(planet, posterior_dir=posterior_dir, format="radvel") # note how we indicate the type of posterior

# Generate point cloud
point_cloud = gen_point_cloud(
    planet, df,
    output_dir=output_dir,
    start_date=start_date,
    end_date=end_date,
    time_interval=10,
    nsamp=100_000,
    out_fname=output,
)

# generate a summary CSV
csv_data = gen_summary_csv(planet, point_cloud, output_dir, output)

# Plot the orbit!
plot_orbital_parameters(
    planet, csv_data,
    params=orbit_params[planet], df_sample=df,
    start_date=start_date, end_date=end_date,
    output_prefix=os.path.join(output_dir, output),
    show_plots=True,
)

Now that we have a point cloud with separations and phase angles at each epoch, we can compute the detection probability: the fraction of orbital posterior draws that Roman could detect within a given integration time budget. This goes beyond the orbital separation and phase angle, as it depends on instrument performance.
Therefore it involves several steps under the hood:

1. **Flux contrast** - For each posterior draw at each epoch, compute the flux ratio using the Batalha+2018 PICASO atmospheric model grid, which accounts for cloud structure, absorption lines, and wavelength-dependent scattering. You can alternatively use a Lambert sphere with a fixed geometric albedo (see following cell for an example) for a more simplified assumption
2. **Integration time** - Feed each draw's separation, contrast, and exozodi into the full `corgietc` noise model to get the required integration time for SNR=5 detection, in both optimistic and conservative scenarios.
3. **Feasible detection probability** - For a given time budget (e.g. 10h or 100h), count the fraction of draws whose integration time falls within that budget. This is the headline metric.
4. **Contrast degradation** - Optionally repeat with a 2× contrast penalty on the optimistic scenario/

The output is a plot showing detection probability, angular separation, phase angle, flux contrast, and integration time distributions over the mission window, with the solar keepout and Galactic Bulge observation windows overlaid.

In [ ]:

# Load Batalha+2018's atmospheric grid
phot_grid = load_photometry_grid(
    photdata_file="grid_model_files/allphotdata.npz",
    dbfile="grid_model_files/AlbedoModels.db",
    star_catalog="grid_model_files/stdata_2025-02-25.p",
)

# Configure it to be used
USE_PICASO = True

def run_planet_band(
    planet, band, obs_mode,
    orbit_props_dir, fname_in,
    int_times_hr, target_snr=5,
    contrast_penalty=2.0,
    n_inttime_samples=10000,
    n_zodis=1, exozodi_inc_override=None,
    show_wfov=False, show_plots=True,
):
    """Run full detection pipeline for one planet+band.
    Returns (scenarios, csv_data, point cloud) or (None, None, None) on failure.
    """
    params = orbit_params[planet]

    # 1) Load point cloud (also used for representative values)
    point_cloud_fname = f"{fname_in}_PointCloud.pkl"
    try:
        pc = load_point_cloud(
            planet, i_dir=orbit_props_dir, fname=point_cloud_fname)
    except FileNotFoundError:
        print('  WARNING: Point cloud not found. Skipping.')
        return None, None, None

    # 2) Get FOV bounds
    try:
        IWA_mas, OWA_mas = get_iwa_owa(planet, obs_mode, target_snr)
        print(f'  IWA={IWA_mas:.1f} mas, OWA={OWA_mas:.1f} mas')
    except Exception as e:
        print(f'  WARNING: Could not get IWA/OWA: {e}')
        return None, None, None

    # 3) Flux contrast
    if USE_PICASO and phot_grid is not None:
        print(f'  Computing flux contrast via picaso (band {band})...')
        pc = compute_contrast_picaso(
            pc, planet, band, phot_grid, seed=1234, show_progress=True)
    else:
        print(f'  Computing flux contrast via Lambert (band {band})...')
        pc = compute_contrast_lambert(pc, band, albedo_dict, albedo_std)

    # 4) Likelihood weights
    lnlike = pc['ln_likelihood']
    if lnlike.ndim == 2:
        lnlike = lnlike[0]
    w = np.exp(lnlike - np.max(lnlike))
    w /= w.sum()

    # 5) Nominal integration times
    print(f'  Calculating integration times (band {band})...')
    pc = calc_integration_time_for_cloud(
        planet, pc,
        target_snr=target_snr, obs_mode=obs_mode, band=band,
        params=params,
        n_inttime_samples=n_inttime_samples,
        IWA_mas=IWA_mas, OWA_mas=OWA_mas,
        max_inttime_hours=1000,
        n_zodis=n_zodis,
        exozodi_inc_override=exozodi_inc_override,
    )

    # 6) Feasible detection probability
    print('  Computing feasible detection probability...')
    scenarios = compute_feasible_pdet(pc, w, max_hours_list=int_times_hr)

    # 7) Degraded (contrast penalty) — opt only
    if contrast_penalty != 1.0:
        pc_deg = copy.copy(pc)
        print(f'  Calculating degraded integration times '
              f'({contrast_penalty:.0f}× penalty)...')
        pc_deg = calc_integration_time_for_cloud(
            planet, pc_deg,
            target_snr=target_snr, obs_mode=obs_mode, band=band,
            params=params,
            n_inttime_samples=n_inttime_samples,
            IWA_mas=IWA_mas, OWA_mas=OWA_mas,
            max_inttime_hours=1000,
            n_zodis=n_zodis,
            exozodi_inc_override=exozodi_inc_override,
            contrast_degradation=contrast_penalty,
        )

        deg_indices = pc_deg['integration_time_sample_indices']
        deg_w = w[deg_indices]
        deg_w /= deg_w.sum()

        for t_hr in int_times_hr:
            arr = pc_deg['integration_time_hours_opt']
            is_feasible = np.isfinite(arr) & (arr <= t_hr)
            pdet = np.average(
                is_feasible.astype(float), axis=1, weights=deg_w)
            label = f'{t_hr}h opt {contrast_penalty:.0f}x'
            scenarios[label] = pdet
            print(f'  {label}: pdet [{pdet.min():.3f}, {pdet.max():.3f}]')

    # 8) Observation windows
    pc = compute_observation_windows(pc, planet)

    # 9) Store feasible pdet for summary CSV
    opt_key = f'{max(int_times_hr)}h opt'
    con_key = f'{max(int_times_hr)}h con'
    if opt_key in scenarios:
        pc['feasible_det_prob_opt'] = scenarios[opt_key]
    if con_key in scenarios:
        pc['feasible_det_prob_con'] = scenarios[con_key]

    # 10) Summary CSV
    fname_csv = f"{fname_in}_Band{band}_contrastComparison"
    csv_data = gen_summary_csv(planet, pc, orbit_props_dir, fname_csv)

    # 11) Plot
    plot_contrast_comparison(
        planet, band, csv_data, scenarios, int_times_hr,
        IWA_mas, OWA_mas,
        orbit_props_dir, fname_in,
        targ_observable=pc.get('targ_observable'),
        GB_not_observable=pc.get('GB_not_observable'),
        contrast_penalty=contrast_penalty,
        target_snr=target_snr,
        n_zodis=n_zodis,
        show_wfov=show_wfov, show_plots=show_plots,
    )

    return scenarios, csv_data, pc
budget_times = [10,100]

scenarios, csv_data, pc = run_planet_band(
    "ups_And_d", band=1, obs_mode="IMG_NFB1_HLC",
    orbit_props_dir=output_dir,
    fname_in=output,
    int_times_hr=budget_times,
    target_snr=5,
    contrast_penalty=2.0,
    show_wfov=False,
)

We can also look at the distribution of integration times across posterior draws at a single epoch. This shows how much observation time each orbital configuration would require, and what fraction of draws fall within a given time budget.

You can either pick a specific date, or let the function find the epoch with the highest detection probability within a date range.

In [ ]:
from roman_plotting_tools import plot_inttime_histogram

# If you want this for a specific date:
#plot_inttime_histogram("ups_And_d", band=1, point_cloud=pc,
#                       target_date='2027-01-15')

# Or for the peak probability date in a given window:
plot_inttime_histogram("ups_And_d", band=1, point_cloud=pc,
                       date_range=('2026-12-01', '2027-06-30'))

In [ ]:
#let's do the version with Lambertian + albedo!
USE_PICASO = False
albedo_dict = {1: 0.3, 3: 0.3, 4: 0.3}
albedo_std = 0.1

scenarios_lambert, csv_data_lambert, pc_lambert = run_planet_band(
    "ups_And_d", band=1, obs_mode="IMG_NFB1_HLC",
    orbit_props_dir=output_dir,
    fname_in=output,
    int_times_hr=[10, 100],
    target_snr=5,
    contrast_penalty=2.0,
    show_wfov=False,
)

In [ ]:

plot_inttime_histogram(
    "ups_And_d", band=1, point_cloud=pc_lambert,
    date_range=('2026-12-01', '2027-06-01'),
    scenario='opt', n_zodis=1,
)